In [8]:
# Use to learn and test the data and run the algorithm GRMMF
# --- Import Modules --- #
import sys, statistics, time, string, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, 'C:/Users/dema2/OneDrive/Desktop/PhD/Tactile-Feedback-Repo/Reflex-Fuzzy-Network')
from RFMN import ReflexFuzzyNeuroNetwork

np.set_printoptions(threshold=5)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, fbeta_score, matthews_corrcoef
from sklearn.utils.multiclass import unique_labels


In [9]:
buffer_size = 149 
data_buffer = []

def median_filter(data):
    data_buffer.append(data)
    if len(data_buffer) > buffer_size:
        data_buffer.pop(0)  
    median_value = np.median(data_buffer)
    return median_value

def simulate_data_input(X):
    original_data = []
    filtered_data = []
    for element in np.nditer(X):
        data_point = element  
        filtered_point = median_filter(data_point)
        original_data.append(data_point)
        filtered_data.append(filtered_point)

    return original_data, filtered_data

In [10]:
# --- Import Iris data, split, and scale --- #
# data = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\Reflex-Fuzzy-Network\\Arduino_train_test_labels.csv')
data = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\SaveModel\\Iris_feedback.csv')
data = data.iloc[:,1:]
X = data.iloc[:, :-1].values
y = data.iloc[:, 4].values

sepal_length = data.iloc[:, :-4].values
sepal_width = data.iloc[:, 1:-3].values
petal_length = data.iloc[:, 2:-2].values
petal_width = data.iloc[:, 3:-1].values

original_data, sepal_length_data = simulate_data_input(sepal_length)
original_data, sepal_width_data = simulate_data_input(sepal_width)
original_data, petal_length_data = simulate_data_input(petal_length)
original_data, petal_width_data = simulate_data_input(petal_width)

sepal_length_data = np.array(sepal_length_data)
sepal_width_data = np.array(sepal_width_data)
petal_length_data = np.array(petal_length_data)
petal_width_data = np.array(petal_width_data)

combined_array_X = np.vstack((sepal_length_data, sepal_width_data, petal_length_data, petal_width_data)).T

print("this is combined array \n", combined_array_X)
print("this is combined array max \n", combined_array_X.max())

print("this is combined array min \n", combined_array_X.min())


X_norm = (combined_array_X-combined_array_X.min())/(combined_array_X.max()-combined_array_X.min())
print("this is X_norm \n", X_norm)


# scaler_min_max = MinMaxScaler(feature_range=(0.00001, .99999))
# X_norm = scaler_min_max.fit_transform(combined_array_X)
# print("this is X_norm \n", X_norm)

this is combined array 
 [[5.1 5.8 3.  4.4]
 [5.  5.8 3.  4.4]
 [4.9 5.8 3.  4.4]
 ...
 [5.8 3.  4.3 1.3]
 [5.8 3.  4.3 1.3]
 [5.8 3.  4.4 1.3]]
this is combined array max 
 5.8
this is combined array min 
 1.3
this is X_norm 
 [[0.84444444 1.         0.37777778 0.68888889]
 [0.82222222 1.         0.37777778 0.68888889]
 [0.8        1.         0.37777778 0.68888889]
 ...
 [1.         0.37777778 0.66666667 0.        ]
 [1.         0.37777778 0.66666667 0.        ]
 [1.         0.37777778 0.68888889 0.        ]]


In [11]:

X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.33, random_state=42)  



print(X_test)

X_train, X_test = X_train.T, X_test.T 
print(X_test)
print(y_test)

nn = ReflexFuzzyNeuroNetwork(gamma=1, theta=.1)
nn.train(X_train, y_train)
print("Model is trained")
# --- Test Network --- #
y_predlr = nn.test(X_test,y_test)

print("done with testings")

[[0.84444444 0.8        0.33333333 0.37777778]
 [0.82222222 1.         0.35555556 0.68888889]
 [0.95555556 0.42222222 0.42222222 0.        ]
 ...
 [0.97777778 0.37777778 0.6        0.        ]
 [0.85555556 0.64444444 0.33333333 0.06666667]
 [0.93333333 0.44444444 0.37777778 0.        ]]
[[0.84444444 0.82222222 0.95555556 ... 0.97777778 0.85555556 0.93333333]
 [0.8        1.         0.42222222 ... 0.37777778 0.64444444 0.44444444]
 [0.33333333 0.35555556 0.42222222 ... 0.6        0.33333333 0.37777778]
 [0.37777778 0.68888889 0.         ... 0.         0.06666667 0.        ]]
[2 1 3 ... 3 2 3]
Model is trained
[2 1 3 ... 3 2 3]
done with testings


In [12]:
# check results
print("confusion_matrix \n", confusion_matrix(y_test, y_predlr), "\n")
print("classification_report \n", classification_report(y_test, y_predlr), "\n")

confusion_matrix 
 [[19  0  0]
 [ 0 15  0]
 [ 0  0 16]] 

classification_report 
               precision    recall  f1-score   support

           1       1.00      1.00      1.00        19
           2       1.00      1.00      1.00        15
           3       1.00      1.00      1.00        16

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50
 



In [13]:
# predict_a = np.array([5.1, 5.8,  3.0, 4.4])
predict_a = np.array([5.8, 3.0,  4.4, 1.3])

print("this is predict_a \n", predict_a)

new_predict_a = predict_a.reshape(1, 4)
print("this is new_predict_a \n", new_predict_a)


# scaler_min_max = MinMaxScaler(feature_range=(0.00001, .99999))
# predict_a_trans = scaler_min_max.fit_transform(new_predict_a)

predict_a_trans = (new_predict_a-combined_array_X.min())/(combined_array_X.max()-combined_array_X.min())
print("this is predict_a_trans \n", predict_a_trans, '\n')


# test = scaler_min_max.fit_transform(combined_array_X)
# print("this is test \n", test)


predict_b = predict_a_trans.ravel()
print(predict_b)








this is predict_a 
 [5.8 3.  4.4 1.3]
this is new_predict_a 
 [[5.8 3.  4.4 1.3]]
this is predict_a_trans 
 [[1.         0.37777778 0.68888889 0.        ]] 

[1.         0.37777778 0.68888889 0.        ]


In [16]:

prediction = nn.predict(predict_b)
prediction = nn.predict([1.00461324 , 0.39409263 , 0.3973099,  -0.06295497])

print(prediction)


3
